# 13a — Hyperparameter Tuning

Corresponds to: Note 13 — Hyperparameter Tuning

**Libraries introduced in this notebook:** No new libraries — `torch` and `torch.nn` introduced in Note 09.

**What this notebook demonstrates:**
1. Learning rate range test — finding a good starting η empirically
2. Batch size effect — comparing training dynamics with small vs large batch
3. Early stopping — implemented from scratch with patience and weight restoration
4. Tuning workflow — applying the Note 13 Part 6 order on a concrete example

In [ ]:
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(42)
print(f'PyTorch version: {torch.__version__}')

## Setup: Dataset and Model

We use a synthetic binary classification dataset throughout this notebook.
Two Gaussian clusters — one per class — with 400 training samples and 100 validation samples.

In [ ]:
rng = np.random.default_rng(0)

def make_dataset(n=500):
    X0 = rng.normal([-1, -1], 0.8, (n // 2, 2))
    X1 = rng.normal([ 1,  1], 0.8, (n // 2, 2))
    X  = np.vstack([X0, X1]).astype(np.float32)
    y  = np.array([0] * (n // 2) + [1] * (n // 2), dtype=np.float32).reshape(-1, 1)
    idx = rng.permutation(n)
    return torch.tensor(X[idx]), torch.tensor(y[idx])

X_train, y_train = make_dataset(400)
X_val,   y_val   = make_dataset(100)

print(f'Training set:   {X_train.shape}, labels {y_train.shape}')
print(f'Validation set: {X_val.shape},   labels {y_val.shape}')

def make_model():
    torch.manual_seed(0)
    model = nn.Sequential(
        nn.Linear(2, 16), nn.ReLU(),
        nn.Linear(16, 8), nn.ReLU(),
        nn.Linear(8,  1), nn.Sigmoid()
    )
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight)
            nn.init.zeros_(m.bias)
    return model

## Part 1: Learning Rate Range Test

Start with a very small η and increase it exponentially over 100 steps.
Record the loss at each step. The good learning rate range is where loss decreases.
A good starting η is one order of magnitude below where loss starts to increase.

In [ ]:
model     = make_model()
criterion = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-7)

eta_start = 1e-7
eta_end   = 1.0
n_steps   = 100
mult      = (eta_end / eta_start) ** (1 / n_steps)

lrs, losses = [], []
model.train()
for step in range(n_steps):
    # Set current lr
    lr = eta_start * (mult ** step)
    for pg in optimizer.param_groups:
        pg['lr'] = lr

    optimizer.zero_grad()
    y_hat = model(X_train)
    loss  = criterion(y_hat, y_train)
    loss.backward()
    optimizer.step()

    lrs.append(lr)
    losses.append(loss.item())

# Print key points
print('Learning Rate Range Test:')
print(f"{'Step':>6}  {'LR':>12}  {'Loss':>8}")
print("-" * 32)
for i in [0, 20, 40, 60, 70, 80, 90, 99]:
    print(f"{i:>6}  {lrs[i]:>12.2e}  {losses[i]:>8.4f}")

min_loss_idx = int(np.argmin(losses))
print(f'\nLowest loss at step {min_loss_idx}: lr = {lrs[min_loss_idx]:.2e}, loss = {losses[min_loss_idx]:.4f}')
print(f'Suggested starting lr: ~{lrs[min_loss_idx] / 10:.2e}  (one order of magnitude below minimum)')

## Part 2: Batch Size Effect

We train the same model with two batch sizes — 8 (small, noisy) and 256 (large, stable) —
and compare validation loss after 50 epochs. Both use the same learning rate.

In [ ]:
def train_epochs(batch_size, lr=0.01, n_epochs=50):
    model     = make_model()
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    val_losses = []

    for epoch in range(n_epochs):
        model.train()
        # Mini-batch loop
        idx = torch.randperm(len(X_train))
        for start in range(0, len(X_train), batch_size):
            batch_idx = idx[start:start + batch_size]
            xb, yb = X_train[batch_idx], y_train[batch_idx]
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val), y_val).item()
        val_losses.append(val_loss)

    return val_losses

torch.manual_seed(1)
val_small = train_epochs(batch_size=8)
torch.manual_seed(1)
val_large = train_epochs(batch_size=256)

print('Validation loss comparison:')
print(f"{'Epoch':>6}  {'Batch=8':>10}  {'Batch=256':>10}")
print("-" * 32)
for ep in [0, 9, 19, 29, 39, 49]:
    print(f"{ep+1:>6}  {val_small[ep]:>10.4f}  {val_large[ep]:>10.4f}")
print(f'\nFinal val loss — batch=8: {val_small[-1]:.4f},  batch=256: {val_large[-1]:.4f}')

## Part 3: Early Stopping

Early stopping monitors validation loss and stops when it has not improved for `patience` epochs.
The best weights are saved and restored at the end.

We implement it from scratch and demonstrate it on the same dataset.

In [ ]:
import copy

def train_with_early_stopping(patience=5, max_epochs=100, lr=0.01, batch_size=32):
    model     = make_model()
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_loss    = float('inf')
    best_weights     = None
    patience_counter = 0
    stopped_at       = max_epochs

    print(f"{'Epoch':>6}  {'Val Loss':>10}  {'Best':>8}  {'Patience':>8}")
    print("-" * 40)

    for epoch in range(1, max_epochs + 1):
        model.train()
        idx = torch.randperm(len(X_train))
        for start in range(0, len(X_train), batch_size):
            batch_idx = idx[start:start + batch_size]
            xb, yb = X_train[batch_idx], y_train[batch_idx]
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val), y_val).item()

        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            best_weights     = copy.deepcopy(model.state_dict())
            patience_counter = 0
            marker = '← best'
        else:
            patience_counter += 1
            marker = ''

        if epoch <= 20 or marker:
            print(f"{epoch:>6}  {val_loss:>10.4f}  {best_val_loss:>8.4f}  {patience_counter:>8}  {marker}")

        if patience_counter >= patience:
            stopped_at = epoch
            break

    # Restore best weights
    model.load_state_dict(best_weights)
    print(f'\nStopped at epoch {stopped_at}. Best val loss: {best_val_loss:.4f}')
    return model

torch.manual_seed(2)
best_model = train_with_early_stopping(patience=5)

## Part 4: Tuning Workflow

Applying the Note 13 Part 6 workflow — one change at a time, in order of impact.
We start from defaults and show the effect of each tuning step.

In [ ]:
def quick_eval(lr, batch_size, dropout_rate=0.0, n_epochs=30, seed=0):
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Linear(2, 16), nn.ReLU(), nn.Dropout(dropout_rate),
        nn.Linear(16, 8), nn.ReLU(), nn.Dropout(dropout_rate),
        nn.Linear(8,  1), nn.Sigmoid()
    )
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight)
            nn.init.zeros_(m.bias)

    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(n_epochs):
        model.train()
        idx = torch.randperm(len(X_train))
        for start in range(0, len(X_train), batch_size):
            b = idx[start:start + batch_size]
            optimizer.zero_grad()
            criterion(model(X_train[b]), y_train[b]).backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_val), y_val).item()
        preds    = (model(X_val) > 0.5).float()
        acc      = (preds == y_val).float().mean().item()
    return val_loss, acc

experiments = [
    ('Step 1 — defaults',              dict(lr=0.001, batch_size=32)),
    ('Step 2 — lr tuned (range test)', dict(lr=0.01,  batch_size=32)),
    ('Step 3 — batch size tuned',      dict(lr=0.01,  batch_size=16)),
    ('Step 4 — dropout added',         dict(lr=0.01,  batch_size=16, dropout_rate=0.2)),
]

print(f"{'Experiment':<38}  {'Val Loss':>9}  {'Val Acc':>8}")
print("-" * 60)
for label, kwargs in experiments:
    val_loss, acc = quick_eval(**kwargs)
    print(f"{label:<38}  {val_loss:>9.4f}  {acc:>8.4f}")

## Summary

This notebook demonstrated:

1. **Learning rate range test** — increase η exponentially over 100 steps and find where loss is minimised. A good starting η is one order of magnitude below the divergence point.
2. **Batch size effect** — small batches (8) produce noisier training but often better final validation loss than large batches (256) at the same learning rate, consistent with Note 13 Part 3.
3. **Early stopping** — patience counter increments when validation loss does not improve, resets when it does. Best weights are saved and restored. Stops training before overfitting without needing to choose epochs in advance.
4. **Tuning workflow** — one change at a time in order: learning rate → batch size → regularization. Each step is evaluated independently so the effect of each change is attributable.